# 💎 Jewelry Sales Prediction — Full Pipeline
**Steps covered:**
1. Load & Name Columns
2. Filter to Purchases
3. Clean Timestamp & Category
4. Drop Useless Columns
5. EDA & Visualisation
6. Outlier Check
7. Aggregate to Monthly Sales
8. Feature Engineering
9. Encode Category
10. Train/Test Split
11. Train 5 Models + Hyperparameter Tuning
12. Evaluate & Compare
13. Feature Importance
14. Save Best Model

## Cell 1 — Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_val_score

print('All libraries imported successfully!')

## Cell 2 — Load & Name Columns

In [ ]:
# Load CSV - no headers exist in the file
df = pd.read_csv('jewelry_combined.csv', header=None)

# Name all columns based on decoded meaning
df.columns = [
    'timestamp',   # col 0 - when the event happened
    'user_id',     # col 1 - who did it
    'session_id',  # col 2 - which browsing session
    'constant',    # col 3 - always 1, useless
    'product_id',  # col 4 - which product
    'category',    # col 5 - earring, ring, pendant etc.
    'event_type',  # col 6 - 0=view, 1=purchase, 2=add to cart
    'price',       # col 7 - price of the item
    'order_id',    # col 8 - order reference
    'gender',      # col 9 - f/m (sparse)
    'color',       # col 10 - red, white, yellow
    'material',    # col 11 - gold, silver, platinum
    'gemstone'     # col 12 - diamond, topaz, pearl etc.
]

print('Shape:', df.shape)
print()
print('Sample:')
print(df.head(3))
print()
print('Dtypes:')
print(df.dtypes)

## Cell 3 — Filter to Actual Purchases Only

In [ ]:
# Check what values exist in event_type
print('event_type value counts:')
print(df['event_type'].value_counts().head(10))
print()
print('Missing in event_type:', df['event_type'].isna().sum())

# Keep only actual purchases (event_type == 1.0)
df_purchases = df[df['event_type'] == 1.0].copy()
df_purchases = df_purchases.dropna(subset=['event_type'])

print()
print('Rows before filtering:', len(df))
print('Rows after filtering :', len(df_purchases))
print('Rows removed         :', len(df) - len(df_purchases))

## Cell 4 — Convert Timestamp & Clean Category

In [ ]:
# Convert timestamp to datetime
df_purchases['timestamp'] = pd.to_datetime(df_purchases['timestamp'], utc=True, errors='coerce')
df_purchases['year']  = df_purchases['timestamp'].dt.year
df_purchases['month'] = df_purchases['timestamp'].dt.month

print('Timestamp sample:')
print(df_purchases[['timestamp', 'year', 'month']].head(3))
print()

# Check category values
print('Category value counts:')
print(df_purchases['category'].value_counts())

# Keep only valid jewelry categories
valid_categories = [
    'jewelry.earring',
    'jewelry.ring',
    'jewelry.pendant',
    'jewelry.bracelet',
    'jewelry.necklace',
    'jewelry.brooch'
]
df_purchases = df_purchases[df_purchases['category'].isin(valid_categories)]

print()
print('Rows after category cleaning:', len(df_purchases))
print()
print('Categories remaining:')
print(df_purchases['category'].value_counts())

## Cell 5 — Drop Useless Columns

In [ ]:
# Drop columns that add no value for monthly sales prediction
columns_to_drop = [
    'constant',   # always 1
    'user_id',    # just an ID
    'session_id', # just an ID
    'order_id',   # just an ID
    'product_id', # just an ID
    'event_type'  # all 1.0 now
]
df_purchases = df_purchases.drop(columns=columns_to_drop)

print('Columns remaining:')
print(df_purchases.columns.tolist())
print()
print('Shape:', df_purchases.shape)
print()
print('Missing values:')
print(df_purchases.isnull().sum())

## Cell 6 — EDA: Exploratory Data Analysis & Visualisation

In [ ]:
# ── EDA 1: Sales count per category ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA — Jewelry Sales Overview', fontsize=16, fontweight='bold')

# Plot 1: Total purchases per category
ax1 = axes[0, 0]
cat_counts = df_purchases['category'].value_counts()
cat_counts.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('Total Purchases per Category')
ax1.set_xlabel('')
ax1.set_xticklabels([c.replace('jewelry.','') for c in cat_counts.index], rotation=45)
ax1.set_ylabel('Count')

# Plot 2: Price distribution per category
ax2 = axes[0, 1]
df_purchases.boxplot(column='price', by='category', ax=ax2)
ax2.set_title('Price Distribution per Category')
ax2.set_xlabel('')
ax2.set_xticklabels([c.replace('jewelry.','') for c in valid_categories], rotation=45)
plt.sca(ax2)
plt.suptitle('')

# Plot 3: Monthly purchases over time (all categories)
ax3 = axes[0, 2]
monthly_total = df_purchases.groupby(['year','month']).size().reset_index(name='count')
monthly_total['date'] = pd.to_datetime(monthly_total[['year','month']].assign(day=1))
ax3.plot(monthly_total['date'], monthly_total['count'], color='steelblue', linewidth=1.5)
ax3.fill_between(monthly_total['date'], monthly_total['count'], alpha=0.2, color='steelblue')
ax3.set_title('Total Monthly Purchases Over Time')
ax3.set_xlabel('')
ax3.set_ylabel('Purchases')
ax3.tick_params(axis='x', rotation=45)

# Plot 4: Sales by month (seasonality)
ax4 = axes[1, 0]
monthly_avg = df_purchases.groupby('month').size() / df_purchases['year'].nunique()
monthly_avg.plot(kind='bar', ax=ax4, color='coral', edgecolor='white')
ax4.set_title('Average Sales by Month (Seasonality)')
ax4.set_xlabel('Month')
ax4.set_ylabel('Avg Purchases')
ax4.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                      'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

# Plot 5: Color distribution
ax5 = axes[1, 1]
color_counts = df_purchases['color'].value_counts()
ax5.pie(color_counts, labels=color_counts.index,
        autopct='%1.1f%%', colors=['#ff6b6b','#f8f9fa','#ffd93d'])
ax5.set_title('Sales by Color')

# Plot 6: Material distribution
ax6 = axes[1, 2]
material_counts = df_purchases['material'].value_counts()
ax6.pie(material_counts, labels=material_counts.index,
        autopct='%1.1f%%', colors=['#ffd700','#c0c0c0','#e5e4e2'])
ax6.set_title('Sales by Material')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA Overview saved!')

In [ ]:
# ── EDA 2: Per category sales trend over time ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Monthly Sales Trend per Category', fontsize=16, fontweight='bold')

colors_plot = ['steelblue','coral','green','purple','orange','brown']

for idx, (category, color) in enumerate(zip(valid_categories, colors_plot)):
    ax = axes[idx // 3][idx % 3]
    cat_data = df_purchases[df_purchases['category'] == category]
    monthly  = cat_data.groupby(['year','month']).size().reset_index(name='sales')
    monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
    
    ax.plot(monthly['date'], monthly['sales'], color=color, linewidth=1.5)
    ax.fill_between(monthly['date'], monthly['sales'], alpha=0.15, color=color)
    ax.set_title(category.replace('jewelry.','').capitalize())
    ax.set_xlabel('')
    ax.set_ylabel('Sales')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Trend plots saved!')

In [ ]:
# ── EDA 3: Price analysis ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Price Analysis', fontsize=14, fontweight='bold')

# Price histogram
ax1 = axes[0]
df_purchases['price'].clip(upper=2000).hist(bins=50, ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('Price Distribution (clipped at $2000)')
ax1.set_xlabel('Price ($)')
ax1.set_ylabel('Count')

# Average price per category
ax2 = axes[1]
avg_price = df_purchases.groupby('category')['price'].mean().sort_values(ascending=False)
avg_price.plot(kind='barh', ax=ax2, color='coral', edgecolor='white')
ax2.set_title('Average Price per Category')
ax2.set_xlabel('Avg Price ($)')
ax2.set_yticklabels([c.replace('jewelry.','') for c in avg_price.index])

plt.tight_layout()
plt.savefig('eda_price.png', dpi=150, bbox_inches='tight')
plt.show()
print('Price analysis saved!')

## Cell 7 — Outlier Check

In [ ]:
# Check price outliers per category
print('Price statistics per category:')
print(df_purchases.groupby('category')['price'].describe().round(2))
print()

# Visualise price outliers
plt.figure(figsize=(10, 5))
df_purchases.boxplot(column='price', by='category')
plt.title('Price Outliers per Category')
plt.suptitle('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Cap extreme price outliers at 99th percentile per category
# This prevents very expensive items from distorting avg_price feature
def cap_outliers(group):
    upper = group['price'].quantile(0.99)
    group['price'] = group['price'].clip(upper=upper)
    return group

df_purchases = df_purchases.groupby('category', group_keys=False).apply(cap_outliers)

print('Price outliers capped at 99th percentile per category')
print('New price stats:')
print(df_purchases.groupby('category')['price'].describe().round(2))

## Cell 8 — Aggregate to Monthly Sales

In [ ]:
# Aggregate purchases by year + month + category
monthly_sales = df_purchases.groupby(
    ['year', 'month', 'category']
).agg(
    sales_count = ('price', 'count'),
    avg_price   = ('price', 'mean')
).reset_index()

# Sort chronologically
monthly_sales = monthly_sales.sort_values(
    ['category', 'year', 'month']
).reset_index(drop=True)

# Remove incomplete December 2021 (only 1 day of real data)
monthly_sales = monthly_sales[
    ~((monthly_sales['year'] == 2021) & (monthly_sales['month'] == 12))
]

print('Shape:', monthly_sales.shape)
print()
print('Months per category:')
print(monthly_sales.groupby('category')['sales_count'].count())
print()
print('Sample:')
print(monthly_sales.head(6))

## Cell 9 — Feature Engineering

In [ ]:
# Sort per category before creating lag features
monthly_sales = monthly_sales.sort_values(
    ['category', 'year', 'month']
).reset_index(drop=True)

# Lag features - last 3 months sales
monthly_sales['lag_1'] = monthly_sales.groupby('category')['sales_count'].shift(1)
monthly_sales['lag_2'] = monthly_sales.groupby('category')['sales_count'].shift(2)
monthly_sales['lag_3'] = monthly_sales.groupby('category')['sales_count'].shift(3)

# Rolling average of last 3 months
monthly_sales['rolling_avg_3'] = (
    monthly_sales.groupby('category')['sales_count']
    .shift(1)
    .rolling(window=3)
    .mean()
    .reset_index(level=0, drop=True)
)

# Rolling average of last 6 months
monthly_sales['rolling_avg_6'] = (
    monthly_sales.groupby('category')['sales_count']
    .shift(1)
    .rolling(window=6)
    .mean()
    .reset_index(level=0, drop=True)
)

# Holiday month flag (Nov, Dec)
monthly_sales['is_holiday_month'] = monthly_sales['month'].isin([11, 12]).astype(int)

# Time index - captures overall growth trend
monthly_sales['time_index'] = monthly_sales.groupby('category').cumcount() + 1

# Quarter
monthly_sales['quarter'] = monthly_sales['month'].apply(lambda x: (x-1)//3 + 1)

# Month over month growth rate (lag1 vs lag2)
monthly_sales['mom_growth'] = (
    (monthly_sales['lag_1'] - monthly_sales['lag_2']) /
    monthly_sales['lag_2'].replace(0, np.nan)
).fillna(0)

# Drop rows with NaN lag values
monthly_sales_clean = monthly_sales.dropna().reset_index(drop=True)

print('Shape after feature engineering:', monthly_sales_clean.shape)
print()
print('All features:')
print(monthly_sales_clean.columns.tolist())
print()
print('Missing values:', monthly_sales_clean.isnull().sum().sum())
print()
print('Sample (earring):')
print(monthly_sales_clean[
    monthly_sales_clean['category']=='jewelry.earring'
].head(5)[['year','month','sales_count','lag_1','rolling_avg_3','time_index','mom_growth']])

## Cell 10 — EDA on Aggregated Data (Correlation Heatmap)

In [ ]:
# Encode category temporarily for correlation
le_temp = LabelEncoder()
temp_df = monthly_sales_clean.copy()
temp_df['category_enc'] = le_temp.fit_transform(temp_df['category'])

numeric_cols = [
    'sales_count', 'avg_price', 'lag_1', 'lag_2', 'lag_3',
    'rolling_avg_3', 'rolling_avg_6', 'is_holiday_month',
    'time_index', 'quarter', 'mom_growth', 'category_enc'
]

plt.figure(figsize=(12, 9))
corr_matrix = temp_df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlation heatmap saved!')
print()
print('Top correlations with sales_count:')
print(corr_matrix['sales_count'].sort_values(ascending=False).round(3))

## Cell 11 — Outlier Check on Monthly Sales

In [ ]:
# Visualise sales_count distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
monthly_sales_clean.boxplot(column='sales_count', by='category')
plt.title('Sales Count Distribution per Category')
plt.suptitle('')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
monthly_sales_clean['sales_count'].hist(bins=40, color='steelblue', edgecolor='white')
plt.title('Overall Sales Count Distribution')
plt.xlabel('Sales Count')
plt.ylabel('Frequency')

plt.tight_layout()
plt.savefig('eda_sales_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print('Sales stats per category:')
print(monthly_sales_clean.groupby('category')['sales_count'].describe().round(1))
print()
print('NOTE: Keeping holiday spikes - they are real seasonal events, not errors')

## Cell 12 — Encode Category & Define Features

In [ ]:
# Encode category as numbers
le = LabelEncoder()
monthly_sales_clean = monthly_sales_clean.copy()
monthly_sales_clean['category_encoded'] = le.fit_transform(
    monthly_sales_clean['category']
)

print('Category encoding:')
for name, code in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {name} → {code}')

# Final feature list
features = [
    'category_encoded',
    'month',
    'year',
    'avg_price',
    'lag_1',
    'lag_2',
    'lag_3',
    'rolling_avg_3',
    'rolling_avg_6',
    'is_holiday_month',
    'time_index',
    'quarter',
    'mom_growth'
]

print()
print('Final feature set:', features)
print('Total features:', len(features))

## Cell 13 — Train/Test Split (Time-based, Per Category)

In [ ]:
# Split per category chronologically - last 6 months = test
train_list = []
test_list  = []

for category in monthly_sales_clean['category'].unique():
    cat_data = monthly_sales_clean[
        monthly_sales_clean['category'] == category
    ].copy()
    
    # Last 6 months = test
    train_list.append(cat_data.iloc[:-6])
    test_list.append(cat_data.iloc[-6:])
    
    print(f'{category}:')
    print(f'  Train: {len(cat_data.iloc[:-6])} rows | Test: {len(cat_data.iloc[-6:])} rows')
    print(f'  Train ends : {cat_data.iloc[-7][["year","month"]].values}')
    print(f'  Test starts: {cat_data.iloc[-6][["year","month"]].values}')

train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

X_train = train_df[features]
y_train = train_df['sales_count']
X_test  = test_df[features]
y_test  = test_df['sales_count']

print()
print('Total training rows:', len(X_train))
print('Total testing rows :', len(X_test))

## Cell 14 — Train 5 Models (Baseline)

In [ ]:
# Define 5 models
models = {
    'Linear Regression'   : LinearRegression(),
    'Ridge Regression'    : Ridge(alpha=1.0),
    'Decision Tree'       : DecisionTreeRegressor(random_state=42),
    'Random Forest'       : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingRegressor(n_estimators=100, random_state=42)
}

baseline_results = {}

print('BASELINE MODEL RESULTS')
print('='*55)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = (abs(y_test.values - y_pred) / y_test.values * 100).mean()
    
    baseline_results[name] = {
        'R2': round(r2, 4),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2)
    }
    
    print(f'\n{name}')
    print(f'  R²   : {r2:.4f}')
    print(f'  MAE  : {mae:.2f}')
    print(f'  RMSE : {rmse:.2f}')
    print(f'  MAPE : {mape:.2f}%')

best_baseline = max(baseline_results, key=lambda x: baseline_results[x]['R2'])
print(f'\nBest baseline model: {best_baseline}')

## Cell 15 — Hyperparameter Tuning (Random Forest & Gradient Boosting)

In [ ]:
# ── Tune Random Forest ───────────────────────────────────────────
print('Tuning Random Forest...')

rf_params = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
rf_grid.fit(X_train, y_train)

print('Best RF params:', rf_grid.best_params_)
print('Best RF CV R²:', round(rf_grid.best_score_, 4))

# ── Tune Gradient Boosting ───────────────────────────────────────
print()
print('Tuning Gradient Boosting...')

gb_params = {
    'n_estimators'  : [100, 200, 300],
    'learning_rate' : [0.05, 0.1, 0.2],
    'max_depth'     : [3, 5, 7],
    'subsample'     : [0.8, 1.0]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
gb_grid.fit(X_train, y_train)

print('Best GB params:', gb_grid.best_params_)
print('Best GB CV R²:', round(gb_grid.best_score_, 4))

## Cell 16 — Final Model Comparison (Tuned vs Baseline)

In [ ]:
# Evaluate tuned models
tuned_models = {
    'Random Forest (tuned)'      : rf_grid.best_estimator_,
    'Gradient Boosting (tuned)'  : gb_grid.best_estimator_
}

all_results = {**baseline_results}

print('TUNED MODEL RESULTS')
print('='*55)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = (abs(y_test.values - y_pred) / y_test.values * 100).mean()
    
    all_results[name] = {
        'R2': round(r2, 4),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2)
    }
    
    print(f'\n{name}')
    print(f'  R²   : {r2:.4f}')
    print(f'  MAE  : {mae:.2f}')
    print(f'  RMSE : {rmse:.2f}')
    print(f'  MAPE : {mape:.2f}%')

# Summary table
print()
print('FULL COMPARISON TABLE')
print('='*55)
results_df = pd.DataFrame(all_results).T
print(results_df.sort_values('R2', ascending=False).to_string())

best_model_name = results_df['R2'].astype(float).idxmax()
print(f'\n🏆 Best overall model: {best_model_name}')
print(f'   R²   : {all_results[best_model_name]["R2"]}')
print(f'   MAPE : {all_results[best_model_name]["MAPE"]}%')

## Cell 17 — Visualise Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')

results_df_plot = pd.DataFrame(all_results).T.astype(float)
model_names     = [m.replace(' (tuned)', '*') for m in results_df_plot.index]

# R² comparison
ax1 = axes[0]
bars = ax1.bar(range(len(results_df_plot)), results_df_plot['R2'],
               color=['steelblue']*5 + ['coral']*2, edgecolor='white')
ax1.set_title('R² Score (higher=better)')
ax1.set_xticks(range(len(model_names)))
ax1.set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax1.set_ylabel('R²')

# MAE comparison
ax2 = axes[1]
ax2.bar(range(len(results_df_plot)), results_df_plot['MAE'],
        color=['steelblue']*5 + ['coral']*2, edgecolor='white')
ax2.set_title('MAE (lower=better)')
ax2.set_xticks(range(len(model_names)))
ax2.set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('MAE')

# MAPE comparison
ax3 = axes[2]
ax3.bar(range(len(results_df_plot)), results_df_plot['MAPE'],
        color=['steelblue']*5 + ['coral']*2, edgecolor='white')
ax3.set_title('MAPE % (lower=better)')
ax3.set_xticks(range(len(model_names)))
ax3.set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('MAPE %')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Model comparison chart saved! (blue=baseline, coral=tuned)')

## Cell 18 — Feature Importance

In [ ]:
# Use best tree-based model for feature importance
best_tree_model = rf_grid.best_estimator_

importance_df = pd.DataFrame({
    'feature'   : features,
    'importance': best_tree_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature Importance (Random Forest Tuned):')
print(importance_df.to_string(index=False))

plt.figure(figsize=(8, 6))
colors_fi = ['coral' if i < 3 else 'steelblue' for i in range(len(importance_df))]
plt.barh(importance_df['feature'], importance_df['importance'],
         color=colors_fi, edgecolor='white')
plt.xlabel('Importance Score')
plt.title('Feature Importance — Random Forest (Tuned)', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance chart saved! (coral = top 3 features)')

## Cell 19 — Actual vs Predicted Visualisation

In [ ]:
# Use best model for final predictions
best_model = rf_grid.best_estimator_
y_pred_final = best_model.predict(X_test)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Actual vs Predicted Sales per Category (Test Set)',
             fontsize=14, fontweight='bold')

for idx, category in enumerate(valid_categories):
    ax = axes[idx // 3][idx % 3]
    
    cat_mask  = test_df['category'] == category
    actual    = y_test[cat_mask].values
    predicted = y_pred_final[cat_mask]
    months    = test_df[cat_mask]['month'].values
    
    x = range(len(actual))
    ax.plot(x, actual,    'o-', color='steelblue', label='Actual',    linewidth=2)
    ax.plot(x, predicted, 's--',color='coral',     label='Predicted', linewidth=2)
    ax.set_title(category.replace('jewelry.','').capitalize())
    ax.set_xlabel('Test Month')
    ax.set_ylabel('Sales')
    ax.legend(fontsize=8)
    ax.set_xticks(x)
    
    mape = (abs(actual - predicted) / actual * 100).mean()
    ax.set_xlabel(f'Avg error: {mape:.1f}%')

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Actual vs Predicted chart saved!')

## Cell 20 — Save Best Model as .pkl

In [ ]:
import pickle

# Save model
with open('jewelry_sales_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save label encoder (needed to decode category predictions)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save feature list (needed for Flask API)
with open('features.pkl', 'wb') as f:
    pickle.dump(features, f)

print('Saved files:')
print('  jewelry_sales_model.pkl  - trained model')
print('  label_encoder.pkl        - category encoder')
print('  features.pkl             - feature list')
print()
print('These 3 files are needed for your Flask API integration.')

# Download all
from google.colab import files
files.download('jewelry_sales_model.pkl')
files.download('label_encoder.pkl')
files.download('features.pkl')
print()
print('Downloads triggered!')